<a href="https://colab.research.google.com/github/kavyansh-1/Sentiment_Analysis/blob/main/Sentiment_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**SENTIMENT ANALYSIS**

In [1]:
# Importing libraries
import os
import json

import pandas as pd
from zipfile import ZipFile
from sklearn.model_selection import train_test_split
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense,Embedding,LSTM
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

**Data Collection - Kaggle API**

In [2]:
kaggle_dictionary=json.load(open("kaggle.json"))

In [3]:
# set up kaggle credentials as environment variables
os.environ['KAGGLE_USERNAME'] = kaggle_dictionary['username']
os.environ['KAGGLE_KEY'] = kaggle_dictionary['key']

In [4]:
#Downloading dataset
!kaggle datasets download -d lakshmi25npathi/imdb-dataset-of-50k-movie-reviews

Dataset URL: https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews
License(s): other
100% 25.7M/25.7M [00:00<00:00, 198MB/s]



In [5]:
!ls

imdb-dataset-of-50k-movie-reviews.zip  kaggle.json  sample_data


In [6]:
#Unzip the dataset file
with ZipFile("imdb-dataset-of-50k-movie-reviews.zip",'r') as zip_ref:
  zip_ref.extractall()

In [7]:
!ls

'IMDB Dataset.csv'			 kaggle.json
 imdb-dataset-of-50k-movie-reviews.zip	 sample_data


**Loading the Dataset**

In [8]:
data=pd.read_csv("IMDB Dataset.csv")

In [9]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   review     50000 non-null  object
 1   sentiment  50000 non-null  object
dtypes: object(2)
memory usage: 781.4+ KB


In [10]:
data['sentiment'].value_counts()

,count
sentiment,
positive,25000
negative,25000


In [11]:
data.replace({"sentiment":{"positive":1,"negative":0}},inplace=True)

/tmp/ipykernel_18882/3877186056.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data.replace({"sentiment":{"positive":1,"negative":0}},inplace=True)


In [12]:
data.sample(5)

,review,sentiment
11208,This quirky and watchable film is the story of...,0
1650,I could not watch more than 10 minutes of this...,0
2273,There is one adjective that describes everythi...,0
44089,Extremely formulaic with cosmic-sized logic ho...,0
16414,Where to begin? Anachronism? High tech cross b...,0


In [13]:
# split data into train and test
train_data,test_data=train_test_split(data,test_size=0.2,random_state=42)

**Data Preprocessing**

In [14]:
#Tokenize Test Data
tokenizer=Tokenizer(num_words=5000)
tokenizer.fit_on_texts(train_data['review'])

X_train=pad_sequences(tokenizer.texts_to_sequences(train_data['review']),maxlen=200)
X_test=pad_sequences(tokenizer.texts_to_sequences(test_data['review']),maxlen=200)

In [15]:
Y_train = train_data['sentiment']
Y_test = test_data['sentiment']

**Building LSTM Model**

In [16]:
from keras.layers import Input

model = Sequential()
model.add(Input(shape=(200,)))
model.add(Embedding(input_dim=5000, output_dim=128))
model.add(LSTM(128, dropout=0.2, recurrent_dropout=0.2))
model.add(Dense(1, activation='sigmoid'))


In [17]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 200, 128)       │       640,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 128)            │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 771,713 (2.94 MB)

 Trainable params: 771,713 (2.94 MB)

 Non-trainable params: 0 (0.00 B)

In [18]:
#Compile the model
model.compile(loss='binary_crossentropy',optimizer='adam',metrics=['accuracy'])

**Training the model**

In [19]:
model.fit(X_train,Y_train,epochs=5,batch_size=64,validation_split=0.2 )

Epoch 1/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 318s 629ms/step - accuracy: 0.7711 - loss: 0.4801 - val_accuracy: 0.8169 - val_loss: 0.4148
Epoch 2/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 309s 618ms/step - accuracy: 0.8448 - loss: 0.3666 - val_accuracy: 0.8531 - val_loss: 0.3404
Epoch 3/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 309s 617ms/step - accuracy: 0.8701 - loss: 0.3109 - val_accuracy: 0.8750 - val_loss: 0.3070
Epoch 4/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 308s 617ms/step - accuracy: 0.8901 - loss: 0.2738 - val_accuracy: 0.8739 - val_loss: 0.3290
Epoch 5/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 324s 620ms/step - accuracy: 0.8960 - loss: 0.2590 - val_accuracy: 0.8756 - val_loss: 0.3109


In [20]:
loass,accuracy=model.evaluate(X_test,Y_test)
print(f"Loss: {loass}")
print(f"Accuracy: {accuracy}")

313/313 ━━━━━━━━━━━━━━━━━━━━ 27s 83ms/step - accuracy: 0.8801 - loss: 0.2957
Loss: 0.2957398295402527
Accuracy: 0.8801000118255615


**Building A Predictive System**

In [21]:
def predict_sentiment(review):
  #Tokenize and pad the review
  tokenized_review = tokenizer.texts_to_sequences([review])
  padded_review = pad_sequences(tokenized_review, maxlen=30)
  prediction=model.predict(padded_review)
  sentiment="positive" if prediction[0][0] >= 0.5 else "negative"
  return sentiment

In [25]:
#Example usage
new_review = "this movie was not that good"
sentiment = predict_sentiment(new_review)
print(f"The sentiment of the review is: {sentiment}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step
The sentiment of the review is: negative


In [27]:
new_review = "this movie was  great"
sentiment = predict_sentiment(new_review)
print(f"The sentiment of the review is: {sentiment}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
The sentiment of the review is: positive
